<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/TOPO_EVO2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install evo2 --no-build-isolation -q

!pip install https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl -q

In [ ]:
!pip show evo2 flash_attn| egrep "Name|Version:"

Name: evo2
Version: 0.6.0
Name: flash_attn
Version: 2.8.3


In [ ]:
!nvidia-smi

Sat Aug  8 12:26:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             50W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
#!/usr/bin/env python3
"""
CATASTROPHIC FORGETTING - FINAL WORKING SOLUTION
Using REAL DNA sequences with modified motifs that the model CAN learn.
"""

import os
import torch
import torch.nn as nn
from torch.optim import AdamW
from evo2 import Evo2
import random
import gc
import numpy as np
from warnings import filterwarnings
from typing import List, Dict, Tuple, Optional

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    MODEL_NAME = 'evo2_7b'

    # DNA Data - REAL sequences with modifications
    DNA_VOCAB = {'A': 0, 'C': 1, 'G': 2, 'T': 3}  # Evo2's DNA vocabulary
    SEQ_LENGTH = 1000  # Long enough to learn patterns
    N_SEQUENCES = 200
    TEST_RATIO = 0.3
    N_TASKS = 4

    # Training
    N_EPOCHS = 20
    LEARNING_RATE = 1e-5  # Lower LR for fine-tuning
    LAMBDA_TOPO = 0.01  # Much lower for real data
    GRADIENT_CLIP = 1.0
    BATCH_SIZE = 8

    SEED = 123
    DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print("="*100)
print("🧬 CATASTROPHIC FORGETTING - REAL DNA SOLUTION")
print("="*100)
print(f"Model: {Config.MODEL_NAME}")
print(f"Sequence length: {Config.SEQ_LENGTH}bp")
print(f"Sequences per task: {Config.N_SEQUENCES}")
print(f"Training epochs: {Config.N_EPOCHS}")
print(f"Learning rate: {Config.LEARNING_RATE}")
print(f"Governance λ: {Config.LAMBDA_TOPO}")
print("="*100 + "\n")

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def cleanup_memory() -> None:
    try:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except:
        pass

def dna_to_tokens(seq: str) -> List[int]:
    """Convert DNA string to token IDs"""
    return [Config.DNA_VOCAB[char] for char in seq]

def tokens_to_dna(tokens: List[int]) -> str:
    """Convert token IDs to DNA string"""
    reverse_vocab = {v: k for k, v in Config.DNA_VOCAB.items()}
    return ''.join(reverse_vocab[t] for t in tokens)

# ============================================================================
# DATA GENERATION - REAL DNA WITH MODIFIED MOTIFS
# ============================================================================

def generate_realistic_dna_sequences(
    n: int,
    motif: str,
    seed: int,
    length: int = Config.SEQ_LENGTH
) -> List[str]:
    """
    Generate DNA sequences with realistic GC content and inserted motifs.
    These are similar to what Evo2 was trained on, but with SPECIFIC motifs.
    """
    random.seed(seed)
    np.random.seed(seed)

    sequences = []
    gc_content = 0.5  # Balanced GC content

    for _ in range(n):
        # Generate realistic DNA with proper GC content
        seq = []
        for _ in range(length):
            if random.random() < gc_content:
                seq.append(random.choice(['G', 'C']))
            else:
                seq.append(random.choice(['A', 'T']))

        # Insert the motif MULTIPLE TIMES (makes it learnable)
        for _ in range(random.randint(5, 15)):
            pos = random.randint(0, length - len(motif))
            seq[pos:pos + len(motif)] = list(motif)

        # Add some local context (realistic flanking sequences)
        for i in range(length):
            if random.random() < 0.1:
                # Local correlations
                if i > 0 and seq[i-1] in ['G', 'C']:
                    seq[i] = random.choice(['G', 'C'])
                else:
                    seq[i] = random.choice(['A', 'T'])

        sequences.append(''.join(seq))

    return sequences

def split_train_test(sequences: List, test_ratio: float = Config.TEST_RATIO) -> Tuple[List, List]:
    random.seed(Config.SEED)
    indices = list(range(len(sequences)))
    random.shuffle(indices)
    split_idx = int(len(sequences) * (1 - test_ratio))
    return [sequences[i] for i in indices[:split_idx]], [sequences[i] for i in indices[split_idx:]]

# ============================================================================
# MODEL LOADING
# ============================================================================

def load_model():
    """Load Evo2 model"""
    evo2_manager = Evo2(Config.MODEL_NAME)
    model = getattr(evo2_manager, 'model', evo2_manager)

    # Convert to nn.Parameter
    for name, module in model.named_modules():
        for param_name, param in module.named_parameters(recurse=False):
            if param is not None:
                new_param = nn.Parameter(param.data.clone().detach(), requires_grad=True)
                setattr(module, param_name, new_param)

    return evo2_manager, model

def save_weights(model):
    return {name: param.data.clone().detach() for name, param in model.named_parameters()}

def restore_weights(model, saved_weights):
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in saved_weights:
                param.data.copy_(saved_weights[name])

# ============================================================================
# TOKENIZATION HELPERS
# ============================================================================

def tokenize_dna(evo2_manager, dna_sequence: str) -> List[int]:
    """Tokenize DNA sequence using Evo2's tokenizer"""
    return evo2_manager.tokenizer.tokenize(dna_sequence)

# ============================================================================
# BASELINE TRAINER
# ============================================================================

class BaselineEvo2Trainer:
    def __init__(self, saved_weights: Dict[str, torch.Tensor], evo2_manager):
        print("[BASELINE] Loading fresh model instance...")
        self.evo2_manager = evo2_manager
        self.model = getattr(evo2_manager, 'model', evo2_manager)
        self.device = Config.DEVICE

        print("Restoring weights...")
        restore_weights(self.model, saved_weights)
        self.model.to(self.device)
        self.model.train()
        print("✅ Baseline ready\n")

    def fine_tune_step(self, dna_sequence: str, optimizer, criterion) -> float:
        self.model.train()
        optimizer.zero_grad()

        token_ids = self.evo2_manager.tokenizer.tokenize(dna_sequence)
        input_ids = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(self.device)

        outputs = self.model(input_ids)
        logits = outputs[0] if isinstance(outputs, tuple) else outputs

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = input_ids[..., 1:].contiguous()

        loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=Config.GRADIENT_CLIP)
        optimizer.step()
        return loss.item()

    def compute_loss(self, dna_sequences: List[str], criterion) -> float:
        self.model.eval()
        losses = []
        with torch.no_grad():
            for seq in dna_sequences:
                token_ids = self.evo2_manager.tokenizer.tokenize(seq)
                input_ids = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(self.device)
                outputs = self.model(input_ids)
                logits = outputs[0] if isinstance(outputs, tuple) else outputs
                shift_logits = logits[..., :-1, :].contiguous()
                shift_labels = input_ids[..., 1:].contiguous()
                loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
                losses.append(loss.item())
        return sum(losses) / len(losses) if losses else 0.0

    def cleanup(self):
        try:
            del self.model
            cleanup_memory()
        except:
            pass

# ============================================================================
# GOVERNANCE TRAINER
# ============================================================================

class GovernedEvo2Trainer:
    def __init__(self, saved_weights: Dict[str, torch.Tensor], evo2_manager):
        print("[GOVERNANCE] Loading fresh model instance...")
        self.evo2_manager = evo2_manager
        self.model = getattr(evo2_manager, 'model', evo2_manager)
        self.device = Config.DEVICE

        print("Restoring weights...")
        restore_weights(self.model, saved_weights)
        self.model.to(self.device)

        self.anchors = {}
        self.captured_states = None
        self.hook_handle = None
        self.anchor_captured = False

        print("Registering governance hook...")
        self._register_governance_hook()
        self.model.train()
        print("✅ Governance ready\n")

    def _register_governance_hook(self):
        def hook(module, input, output):
            hidden_states = output[0] if isinstance(output, tuple) else output
            self.captured_states = hidden_states
            if not self.anchor_captured:
                self.anchors['blocks.28'] = hidden_states.detach().clone()
                self.anchor_captured = True
                print(f"✅ Anchor captured: {hidden_states.shape}")

        for name, module in self.model.named_modules():
            if name == 'blocks.28':
                self.hook_handle = module.register_forward_hook(hook)
                print(f"✅ Hook registered on {name}")
                return
        raise RuntimeError("blocks.28 not found!")

    def capture_anchors_before_training(self, anchor_sequence: str):
        print("Capturing anchor states...")
        self.model.eval()
        with torch.no_grad():
            token_ids = self.evo2_manager.tokenizer.tokenize(anchor_sequence)
            input_ids = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(self.device)
            _ = self.model(input_ids)
        self.anchor_captured = True
        print(f"✅ Anchor captured")
        self.model.train()

    def fine_tune_step(self, dna_sequence: str, optimizer, criterion, lambda_topo=Config.LAMBDA_TOPO):
        self.model.train()
        optimizer.zero_grad()

        token_ids = self.evo2_manager.tokenizer.tokenize(dna_sequence)
        input_ids = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(self.device)

        outputs = self.model(input_ids)
        logits = outputs[0] if isinstance(outputs, tuple) else outputs

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = input_ids[..., 1:].contiguous()
        task_loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        topo_loss = torch.tensor(0.0, device=self.device)
        if self.captured_states is not None and 'blocks.28' in self.anchors:
            anchor = self.anchors['blocks.28']
            current = self.captured_states
            if anchor.shape == current.shape:
                # L2 constraint
                deviation = torch.norm(current - anchor, p=2, dim=-1)
                topo_loss = lambda_topo * torch.mean(deviation)

        total_loss = task_loss + topo_loss
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=Config.GRADIENT_CLIP)
        optimizer.step()
        self.captured_states = None
        return task_loss.item(), topo_loss.item()

    def compute_loss(self, dna_sequences: List[str], criterion) -> float:
        self.model.eval()
        losses = []
        with torch.no_grad():
            for seq in dna_sequences:
                token_ids = self.evo2_manager.tokenizer.tokenize(seq)
                input_ids = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(self.device)
                outputs = self.model(input_ids)
                logits = outputs[0] if isinstance(outputs, tuple) else outputs
                shift_logits = logits[..., :-1, :].contiguous()
                shift_labels = input_ids[..., 1:].contiguous()
                loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
                losses.append(loss.item())
        return sum(losses) / len(losses) if losses else 0.0

    def cleanup(self):
        try:
            if self.hook_handle is not None:
                self.hook_handle.remove()
            del self.model
            cleanup_memory()
        except:
            pass

# ============================================================================
# MAIN EXPERIMENT
# ============================================================================

class CatastrophicForgettingExperiment:
    def __init__(self):
        self.saved_weights = None
        self.criterion = nn.CrossEntropyLoss()
        self.evo2_manager = None

    def run(self):
        try:
            set_seed(Config.SEED)

            # Load model once and save weights
            print("[INIT] Loading model...")
            self.evo2_manager, model = load_model()
            self.saved_weights = save_weights(model)
            print(f"✅ Saved {len(self.saved_weights)} parameters\n")
            del model
            cleanup_memory()

            # Generate data - DIFFERENT motifs per task
            print("[DATA] Generating DNA sequences with different motifs...\n")
            data = {}
            motifs = ['TATATATA', 'CGCGCGCG', 'AAAAATTTT', 'GCCGCCGC']

            for task_id, motif in enumerate(motifs, 1):
                sequences = generate_realistic_dna_sequences(
                    Config.N_SEQUENCES, motif, Config.SEED + task_id * 1000
                )
                train_set, test_set = split_train_test(sequences)
                data[f'task_{task_id}'] = {
                    'train': train_set,
                    'test': test_set,
                    'motif': motif
                }
                print(f"Task {task_id} ({motif}): {len(train_set)} train, {len(test_set)} test")
            print()

            # === BASELINE ===
            print("\n" + "="*100)
            print("BASELINE (NO GOVERNANCE)")
            print("="*100 + "\n")

            trainer = BaselineEvo2Trainer(self.saved_weights, self.evo2_manager)
            optimizer = AdamW(trainer.model.parameters(), lr=Config.LEARNING_RATE)

            print(f"Training on Task 1 ({data['task_1']['motif']})...")
            for epoch in range(Config.N_EPOCHS):
                total_loss = 0
                for seq in data['task_1']['train']:
                    loss = trainer.fine_tune_step(seq, optimizer, self.criterion)
                    total_loss += loss
                if (epoch + 1) % 5 == 0:
                    avg_loss = total_loss / len(data['task_1']['train'])
                    print(f"  Epoch {epoch+1}/{Config.N_EPOCHS}, Loss: {avg_loss:.4f}")

            pre_loss = trainer.compute_loss(data['task_1']['test'], self.criterion)
            print(f"\nTask 1 Pre-adaptation Loss: {pre_loss:.4f}")

            # Train on other tasks
            for task_id in range(2, Config.N_TASKS + 1):
                print(f"\nTraining on Task {task_id} ({data[f'task_{task_id}']['motif']})...")
                for epoch in range(Config.N_EPOCHS // 2):
                    for seq in data[f'task_{task_id}']['train']:
                        trainer.fine_tune_step(seq, optimizer, self.criterion)
                    if (epoch + 1) % 5 == 0:
                        print(f"  Epoch {epoch+1}/{Config.N_EPOCHS//2} complete")

            post_loss = trainer.compute_loss(data['task_1']['test'], self.criterion)
            baseline_forgetting = ((post_loss - pre_loss) / pre_loss * 100)
            print(f"\nTask 1 Post-adaptation Loss: {post_loss:.4f}")
            print(f"Catastrophic Forgetting: {baseline_forgetting:+.1f}%\n")

            trainer.cleanup()
            del optimizer
            cleanup_memory()

            # === GOVERNANCE ===
            print("\n" + "="*100)
            print("GOVERNANCE (WITH ANCHOR)")
            print("="*100 + "\n")

            trainer = GovernedEvo2Trainer(self.saved_weights, self.evo2_manager)
            optimizer = AdamW(trainer.model.parameters(), lr=Config.LEARNING_RATE)

            # Capture anchor
            trainer.capture_anchors_before_training(data['task_1']['train'][0])

            print(f"Training on Task 1 ({data['task_1']['motif']}) with anchor lock...")
            for epoch in range(Config.N_EPOCHS):
                task_sum = 0
                topo_sum = 0
                for seq in data['task_1']['train']:
                    task_loss, topo_loss = trainer.fine_tune_step(seq, optimizer, self.criterion)
                    task_sum += task_loss
                    topo_sum += topo_loss
                if (epoch + 1) % 5 == 0:
                    avg_task = task_sum / len(data['task_1']['train'])
                    avg_topo = topo_sum / len(data['task_1']['train'])
                    print(f"  Epoch {epoch+1}/{Config.N_EPOCHS}, Task: {avg_task:.4f}, Topo: {avg_topo:.4f}")

            pre_loss = trainer.compute_loss(data['task_1']['test'], self.criterion)
            print(f"\nTask 1 Pre-adaptation Loss: {pre_loss:.4f}")

            # Train on other tasks with constraint
            for task_id in range(2, Config.N_TASKS + 1):
                print(f"\nTraining on Task {task_id} ({data[f'task_{task_id}']['motif']}) with constraint...")
                for epoch in range(Config.N_EPOCHS // 2):
                    for seq in data[f'task_{task_id}']['train']:
                        trainer.fine_tune_step(seq, optimizer, self.criterion)
                    if (epoch + 1) % 5 == 0:
                        print(f"  Epoch {epoch+1}/{Config.N_EPOCHS//2} complete")

            post_loss = trainer.compute_loss(data['task_1']['test'], self.criterion)
            governance_forgetting = ((post_loss - pre_loss) / pre_loss * 100)
            print(f"\nTask 1 Post-adaptation Loss: {post_loss:.4f}")
            print(f"Catastrophic Forgetting: {governance_forgetting:+.1f}%\n")

            trainer.cleanup()
            del optimizer
            cleanup_memory()

            # === RESULTS ===
            print("\n" + "="*100)
            print("FINAL RESULTS")
            print("="*100 + "\n")

            improvement = baseline_forgetting - governance_forgetting

            print("╔════════════════════════════════════════════════════════════════════════════════════════╗")
            print("║              CATASTROPHIC FORGETTING: BASELINE vs GOVERNANCE                         ║")
            print("╠════════════════════════════════════════════════════════════════════════════════════════╣")
            print("║                                                                                        ║")
            print("║  BASELINE (NO Governance)                                                             ║")
            print(f"║    Pre-adaptation:  {baseline_forgetting:.4f}                                                     ║")
            print(f"║    Post-adaptation: {governance_forgetting:.4f}                                                     ║")
            print(f"║    Forgetting:      {baseline_forgetting:+.1f}%                                                         ║")
            print("║                                                                                        ║")
            print("║  GOVERNANCE (blocks.28 Anchor)                                                        ║")
            print(f"║    Pre-adaptation:  {pre_loss:.4f}                                                     ║")
            print(f"║    Post-adaptation: {post_loss:.4f}                                                     ║")
            print(f"║    Forgetting:      {governance_forgetting:+.1f}%                                                         ║")
            print("║                                                                                        ║")
            print("╠════════════════════════════════════════════════════════════════════════════════════════╣")
            print(f"║  Improvement: {improvement:+.1f} percentage points                                                 ║")

            if improvement > 20:
                print("║  ✅✅✅ GOVERNANCE SIGNIFICANTLY REDUCES CATASTROPHIC FORGETTING                       ║")
            elif improvement > 10:
                print("║  ✅ GOVERNANCE REDUCES CATASTROPHIC FORGETTING                                      ║")
            elif improvement > 5:
                print("║  ⚠️  MODEST IMPROVEMENT - Try adjusting hyperparameters                            ║")
            else:
                print("║  ❌ NO IMPROVEMENT - Check if model is learning                                    ║")
            print("║                                                                                        ║")
            print("╚════════════════════════════════════════════════════════════════════════════════════════╝")

            print("\n" + "="*100)
            print("CONFIGURATION:")
            print(f"  ✓ Model: {Config.MODEL_NAME}")
            print(f"  ✓ Data: Realistic DNA with inserted motifs")
            print(f"  ✓ Sequence length: {Config.SEQ_LENGTH}bp")
            print(f"  ✓ Motifs: {', '.join(motifs)}")
            print(f"  ✓ Training epochs: {Config.N_EPOCHS}")
            print(f"  ✓ Learning rate: {Config.LEARNING_RATE}")
            print(f"  ✓ Governance λ: {Config.LAMBDA_TOPO}")
            print("="*100 + "\n")

        except Exception as e:
            print(f"\n❌ ERROR: {e}")
            import traceback
            traceback.print_exc()
        finally:
            cleanup_memory()
            print("\n✅ Experiment complete")

if __name__ == "__main__":
    experiment = CatastrophicForgettingExperiment()
    experiment.run()

🧬 CATASTROPHIC FORGETTING - REAL DNA SOLUTION
Model: evo2_7b
Sequence length: 1000bp
Sequences per task: 200
Training epochs: 20
Learning rate: 1e-05
Governance λ: 0.01

[INIT] Loading model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt




  0%|          | 0/32 [00:00<?, ?it/s]

 12%|█▎        | 4/32 [00:00<00:00, 36.89it/s]

100%|██████████| 32/32 [00:00<00:00, 121.08it/s]


Extra keys in state_dict: {'blocks.4.projections._extra_state', 'blocks.16.mixer.mixer.filter.t', 'blocks.0.projections._extra_state', 'blocks.23.projections._extra_state', 'blocks.13.mixer.mixer.filter.t', 'blocks.27.projections._extra_state', 'blocks.31.mixer.dense._extra_state', 'blocks.17.mixer.attn._extra_state', 'blocks.9.mixer.mixer.filter.t', 'blocks.8.projections._extra_state', 'blocks.11.projections._extra_state', 'unembed.weight', 'blocks.2.mixer.mixer.filter.t', 'blocks.6.projections._extra_state', 'blocks.5.projections._extra_state', 'blocks.25.projections._extra_state', 'blocks.13.projections._extra_state', 'blocks.20.mixer.mixer.filter.t', 'blocks.23.mixer.mixer.filter.t', 'blocks.7.projections._extra_state', 'blocks.12.projections._extra_state', 'blocks.18.projections._extra_state', 'blocks.16.projections._extra_state', 'blocks.26.projections._extra_state', 'blocks.22.projections._extra_state', 'blocks.15.projections._extra_state', 'blocks.24.mixer.attn._extra_state', '